# Happiness in the music we listen to, and how we respond to it
### By Quinn Epstein
---
## Setup for graphing

In [39]:
import plotly.graph_objects as go;
import opendatasets as od;
import pandas as pd;

od.download('https://www.kaggle.com/datasets/ektanegi/spotifydata-19212020');
od.download('https://www.kaggle.com/datasets/gauthamvijayaraj/spotify-tracks-dataset-updated-every-week');
od.download('https://www.kaggle.com/datasets/khushikyad001/world-happiness-report');

spotifyLegacy = pd.read_csv('spotifydata-19212020/data.csv');
spotifyModern = pd.read_csv('spotify-tracks-dataset-updated-every-week/spotify_tracks.csv');

# Normalize headers
spotifyModern.columns = spotifyModern.columns.str.lower().str.strip();

# fixing inconsistent column names from data
for feature in ['valence', 'danceability']:
    if feature not in spotifyModern.columns:
        match = next((c for c in spotifyModern.columns if feature[:4] in c), None);
        if match: spotifyModern.rename(columns={match: feature}, inplace=True);

if 'year' not in spotifyModern.columns:
    dateCol = next((c for c in spotifyModern.columns if 'date' in c), None);
    spotifyModern['year'] = pd.to_datetime(spotifyModern[dateCol], errors='coerce').dt.year if dateCol else pd.NA;

# saving needed columns 
cols = ['year', 'valence', 'danceability'];
spotifyStats = pd.concat([spotifyLegacy[cols], spotifyModern[cols]], ignore_index=True);

spotifyStats = spotifyStats[(spotifyStats['year'] >= 1921) & (spotifyStats['year'] <= 2024)];
spotifyStats.dropna(subset=['year', 'valence', 'danceability'], inplace=True);

# score normalizing 
happinessScores = pd.read_csv('world-happiness-report/world_happiness_report.csv');
happinessScores.rename(columns={'Happiness_Score': 'Happiness Score'}, inplace=True);
happinessScores['NormalizedScores'] = happinessScores['Happiness Score'] / 10;

# group by year 
yearlyMusic = spotifyStats.groupby('year')[['valence', 'danceability']].mean().reset_index();
yearlyHappiness = happinessScores.groupby('Year')['NormalizedScores'].mean().reset_index();

Skipping, found downloaded files in ".\spotifydata-19212020" (use force=True to force download)
Skipping, found downloaded files in ".\spotify-tracks-dataset-updated-every-week" (use force=True to force download)
Skipping, found downloaded files in ".\world-happiness-report" (use force=True to force download)


---
## First im going to make two graphs showing happiness overtime 

In [40]:
happinessGraph = go.Figure();

happinessGraph.add_trace(go.Scatter(
    x = yearlyMusic['year'],
    y = yearlyMusic['valence'],
    mode = 'lines+markers',
    name = 'Music Happiness (Valence)',
    line = dict(color = '#785EF0', width = 3)));

happinessGraph.update_layout(
    title = 'Happiness of Music (1921 - 2024)',
    xaxis_title = 'Year',
    yaxis_title = 'Music Happiness (0.0 - 1.0)',
    template = 'ggplot2',
    hovermode = 'x unified',
    xaxis = dict(range = [1921, 2027])
);

happinessGraph.show();

## Plus World Happiness Data

In [41]:
happinessGraph.add_trace(go.Scatter(
    x = yearlyHappiness['Year'],
    y = yearlyHappiness['NormalizedScores'],
    mode = 'lines+markers',
    name = 'Global Happiness Score',
    line = dict(color = '#FE6100', width = 4, dash = 'dash')));

happinessGraph.update_layout(
    title = 'Music Happiness vs Global Happiness',
    yaxis_title = 'Happiness Score (0.0 - 1.0)',
    # Zooming in on the 2000s where we have the most correlation data
    xaxis = dict(range = [2000, 2027])
);

happinessGraph.show();

## We now exapand to the full picture, showing danceability vs valence, and how it changes year over year 

In [42]:
#add danceability line
happinessGraph.add_trace(go.Scatter(
    x = yearlyMusic['year'],
    y = yearlyMusic['danceability'],
    mode = 'lines+markers',
    name = 'Danceability',
    line = dict(color = '#FFB000', width = 3, dash = 'dot')
));

happinessGraph.update_layout(
    title = 'Relationship Between Music Vibe and Danceability',
    legend = dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    xaxis = dict(range = [1921, 2027])
);

happinessGraph.show();


# add second graph
yearlyMusic['Vibe_Gap'] = yearlyMusic['danceability'] - yearlyMusic['valence'];

colors = ['#FFB000' if val > 0 else '#785EF0' for val in yearlyMusic['Vibe_Gap']];

gapGraph = go.Figure();

gapGraph.add_trace(go.Bar(
    x=yearlyMusic['year'],
    y=yearlyMusic['Vibe_Gap'],
    marker_color=colors,
    name='Danceability vs. Valence Gap'
));

gapGraph.update_layout(
    title='Is Music More Happy or Danceable Year by Year?',
    xaxis_title='Year',
    yaxis_title='Gap (Danceability - Happiness)',
    template='ggplot2',
    shapes=[dict(type='line', y0=0, y1=0, x0=1921, x1=2024, line=dict(color='black', width=2))]
);

gapGraph.show();

## World Event Context Added

In [43]:
wars = [
    {'start': 1939, 'end': 1945, 'name': 'WWII', 'pos': 'top left'},
    {'start': 1947, 'end': 1991, 'name': 'Cold War', 'pos': 'bottom inside'},
    {'start': 1955, 'end': 1975, 'name': 'Vietnam', 'pos': 'top left'},
    {'start': 1990, 'end': 1991, 'name': 'Gulf War', 'pos': 'top right'},
    {'start': 2001, 'end': 2021, 'name': 'War on Terror', 'pos': 'top left'}
];

for graph in [happinessGraph, gapGraph]:
    for war in wars:
        graph.add_vrect(
            x0=war['start'], x1=war['end'],
            fillcolor="#888888", # Neutral gray to let the main data colors pop
            opacity=0.15, # Keeping it transparent so data is still visible
            layer="below", # Forces the shading behind the data lines	
            line_width=1,
            line_dash="dot",
            annotation_text=war['name'],
            annotation_position=war['pos'],
            annotation_font_size=11,
            annotation_font_color="#666666"
        );

# Re-render both updated graphs
happinessGraph.show();
gapGraph.show();

## this is here solely for the joke 

In [44]:
# Grabbing the 2004 value to anchor the arrow
valenceScore2004 = yearlyMusic[yearlyMusic['year'] == 2004]['valence'].values[0];

# add a little joke
happinessGraph.add_annotation(
    x = 2004,
    y = valenceScore2004,
    text = "Year I was born",
    showarrow = True,
    arrowhead = 2,
    arrowcolor = "#FA4D56", 
    ax = 0,
    ay = 60, 
    font = dict(size = 11, color = "#FA4D56")
);

happinessGraph.show();
gapGraph.show();

## Investigates one of the most recent spikes that doesnt have an obvious reason

In [45]:
yearlyMusic['Valence_Change'] = yearlyMusic['valence'].diff();

# Zoom in on the decade surrounding the spike
zoomWindow = yearlyMusic[(yearlyMusic['year'] >= 2008) & (yearlyMusic['year'] <= 2018)].copy();

colors = ['#785EF0' if val > 0 else '#888888' for val in zoomWindow['Valence_Change']];

cliffGraph = go.Figure();

cliffGraph.add_trace(go.Bar(
    x=zoomWindow['year'],
    y=zoomWindow['Valence_Change'],
    marker_color=colors,
    text=zoomWindow['Valence_Change'].round(3),
    textposition='auto'
));

cliffGraph.update_layout(
    title='The 2013-2014 Vibe Shift: Year-over-Year Momentum in Music Happiness',
    xaxis_title='Year',
    yaxis_title='Change in Valence Score (vs. Previous Year)',
    template='ggplot2',
    shapes=[dict(type='line', y0=0, y1=0, x0=2007.5, x1=2018.5, line=dict(color='black', width=1))]
);

val_2013 = zoomWindow[zoomWindow['year'] == 2013]['Valence_Change'].values[0];
val_2014 = zoomWindow[zoomWindow['year'] == 2014]['Valence_Change'].values[0];

# 2013 label
cliffGraph.add_annotation(
    x=2013,
    y=val_2013,
    text="The 'Happy' Peak<br>(Pharrell, EDM Boom)",
    showarrow=True,
    arrowhead=2,
    arrowcolor="#666666",
    ax=-40,
    ay=-40
);

# 2014 label
cliffGraph.add_annotation(
    x=2014,
    y=val_2014,
    text="The Streaming &<br>Smartphone Shift",
    showarrow=True,
    arrowhead=2,
    arrowcolor="#FE6100",
    ax=40,
    ay=40
);

cliffGraph.show();

## Finally, shows an interesting graph of what current music looks like, and how much fits into what category 

In [46]:
modernTracks = spotifyStats[spotifyStats['year'] >= 2014]

ibm_colorscale = [
    [0.0, '#785EF0'],
    [0.5, '#FE6100'],
    [1.0, '#FFB000']  
];

densityGraph = go.Figure();

densityGraph.add_trace(go.Histogram2dContour(
    x=modernTracks['valence'],
    y=modernTracks['danceability'],
    colorscale=ibm_colorscale,
    reversescale=False,
    contours=dict(showlines=False) 
));

densityGraph.update_layout(
    title='Where Modern Music Lives (2014 - 2024)',
    xaxis_title='Valence (Sad -> Happy)',
    yaxis_title='Danceability (Low -> High)',
    template='ggplot2', 
    width=700, height=700,
    xaxis=dict(range=[0, 1]),
    yaxis=dict(range=[0, 1])
);

densityGraph.add_hline(y=0.5, line_width=1, line_dash="dash", line_color="white", opacity=0.3);
densityGraph.add_vline(x=0.5, line_width=1, line_dash="dash", line_color="white", opacity=0.3);

densityGraph.show();